In [0]:
import sys
import importlib

SRC_PATH = "/Workspace/Users/ariamostajeran99@gmail.com/stock_project_V2/stock-mlops-databricks/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

import config
import dataset_builder

importlib.reload(config)
importlib.reload(dataset_builder)

from config import FEATURE_TABLE_NAME, ML_DATASET_TABLE_NAME, TARGET_HORIZON_DAYS, TARGET_RETURN_THRESHOLD
from dataset_builder import DatasetBuilder

In [0]:
features_df = spark.table(FEATURE_TABLE_NAME).toPandas()

print("Feature store rows:", len(features_df))
print("Feature store cols:", len(features_df.columns))
features_df.head()

In [0]:
print(features_df[features_df["Ticker"] == "AAPL"].head(3))
print(features_df[features_df["Ticker"] == "NVDA"].head(3))

In [0]:
builder = DatasetBuilder(
    target_horizon_days=TARGET_HORIZON_DAYS,
    target_return_threshold=TARGET_RETURN_THRESHOLD
)
dataset_with_targets = builder.add_targets(features_df)
dataset_with_targets = builder.add_naive_baseline(dataset_with_targets)

print("Rows after targets:", len(dataset_with_targets))
dataset_with_targets[["Date", "Ticker", "Close", "future_close", "future_return", "target_class", "naive_prediction"]].head(10)

In [0]:
ml_df = builder.build_ml_dataset(dataset_with_targets)
builder.validate_ml_dataset(ml_df)

print("ML dataset rows:", len(ml_df))
print("ML dataset cols:", len(ml_df.columns))
ml_df.head()

In [0]:
feature_cols = builder.get_feature_columns(ml_df)

print("Number of feature columns:", len(feature_cols))
print(feature_cols[:30])

In [0]:
target_summary = (
    ml_df["target_class"]
    .value_counts(normalize=True)
    .sort_index()
)

print(target_summary)

In [0]:
print("Min date:", ml_df["Date"].min())
print("Max date:", ml_df["Date"].max())
print("Tickers:", ml_df["Ticker"].nunique())
print(sorted(ml_df["Ticker"].unique()))

In [0]:
spark.createDataFrame(ml_df) \
    .write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(ML_DATASET_TABLE_NAME)

In [0]:
import numpy as np

numeric_cols = ml_df.select_dtypes(include=[np.number]).columns.tolist()

inf_counts = np.isinf(ml_df[numeric_cols]).sum().sum()
nan_counts = ml_df[numeric_cols].isna().sum().sum()

print("Total inf values in numeric columns:", inf_counts)
print("Total NaN values in numeric columns:", nan_counts)

In [0]:
print(ml_df["target_class"].value_counts(normalize=True).sort_index())
print(ml_df["naive_prediction_class"].value_counts(normalize=True).sort_index())